## Notebook Name: Conservation Areas Injest
**Medallion Layer: Bronze**  

**Purpose:** Ingest the Raw Data of the Conservation Areas that shows the point locations 

**Author:** Matthew Kristanto  

**Date Created:** 4/03/26  

**Last Modified:** 5/03/26  

**Notes:**
- Injesting the raw CSV Files from Microsoft Azure Data Lake Storage
- Does not handle missing or Null Values
- Did Renaming of some of the Column Names which had spaces between them


In [0]:
storage_account_name = "staccbirddata"  
storage_account_key = dbutils.secrets.get(scope="bird-data", key="storage-account-key")

csv_file = "bird_conservation_areas_20260304_120500.csv"


In [0]:
### Read the CSV file
df_bronze = spark.read.format("csv").option("header", "true").option("multiline", "true").load(
    f"wasbs://bronze@{storage_account_name}.blob.core.windows.net/{csv_file}"
)

In [0]:
### ------------------------------------CHECK----------------------------------
### Inspect that the data has been retrieved.

display(df_bronze)

In [0]:
df_bronze.count()

In [0]:
### Create the Schema in the Unity Catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [0]:
### Rename the Columns that had spacing between them
from pyspark.sql.functions import col

def rename_columns(df):  
  new_columns = []

  for c in df.columns:
    new_name = c.replace(" ", "_")
    new_columns.append(col(c).alias(new_name))
  
  ## Unpack the list, so it is not treated as just a single column
  df_bronze_nospace = df.select(*new_columns)
  return df_bronze_nospace

df_bronze_nospace = rename_columns(df_bronze)


In [0]:
(df_bronze_nospace.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.bronze_bird_conservation_areas"))

In [0]:
%sql
SELECT * FROM bronze.bronze_bird_conservation_areas

In [0]:
%sql
--  Then write the Delta Table in the Catalog

-- CREATE OR REFRESH STREAMING TABLE bronze.bronze_conservation_areas
-- COMMENT "Ingest the Raw Conservation Areas CSV Files from Cloud Storage"
-- TBLPROPERTIES (
--   "quality" = "bronze",
--   -- Prevent full refreshes on the table
--   "pipelines.reset.allowed" = FALSE
-- )
-- AS
-- SELECT
--   *,
--   current_timestamp() AS processing_time
-- FROM STREAM (bronze_source)